### *Decision Tree - Binary tree to classify DOG - Cat*

In [22]:
import numpy as np
import matplotlib.pyplot as plt

### *Entropy - Use to calculate purerity of  object in data:*
$$H(S) = -\sum p_i \log_2(p_i)$$
Example: 
- In a group have all DOG (Very purerity): $$Entropy = 0$$ 
- In a group have half is DOG and half of CAT (Not Purerity): $$Entropy = 1$$ 

In [23]:
def calculate_entropy(y):
    hist = np.bincount(y)

    ps = hist / len(y)
    return -np.sum([p * np.log2(p) for p in ps if p >0])

### *Infomation Gain - If i split base on feature A, Will I collect data is more purerity??*

In [24]:
def information_gain(y, x_col, threshold):
    parent_entropy = calculate_entropy(y)

    left_idxs = np.argwhere(x_col <= threshold).flatten()
    right_idxs = np.argwhere(x_col > threshold).flatten()

    if len(left_idxs) == 0 or len(right_idxs) == 0:
        return 0
    
    n = len(y)
    e_l, e_r = calculate_entropy(y[left_idxs]), calculate_entropy(y[right_idxs])
    child_entropy = e_l * len(left_idxs) / n + e_r * len(right_idxs)/n
    return parent_entropy - child_entropy

### *Best split - Find features have the best information to gain and split*

In [25]:
def best_split(X, y, n_features):
    best_gain = -1
    split_idx, split_thresh = None, None

    for i in range(n_features):
        threshold = np.unique(X[:, i])
        for t in threshold:
            gain = information_gain(y, X[:, i], t)
            if gain > best_gain:
                best_gain, split_idx, split_thresh = gain, i, t 
    return split_idx, split_thresh

### *Tree grow down - Combine to classify*

In [26]:
def train_tree(X, y, max_depth=5):
    n_samples, n_features = X.shape
    tree = {}

    node_id_counter = 0

    stack = [(X, y, 0, node_id_counter)]

    while stack:
        curr_X, curr_y, depth, curr_id = stack.pop()

        if depth >= max_depth or len(np.unique(curr_y)) <= 1:
            leaf_value = np.argmax(np.bincount(curr_y)) if len(curr_y) > 0 else 0
            tree[curr_id] = {'value': leaf_value}
            continue

        best_feature, best_thresh = best_split(curr_X, curr_y, n_features)

        if best_feature is None:
            tree[curr_id] = {'value': np.argmax(np.bincount(curr_y))}
            continue

        left_id, right_id = node_id_counter + 1, node_id_counter + 2
        node_id_counter += 2

        tree[curr_id] = {
            'feature': best_feature,
            'threshold': best_thresh,
            'left': left_id,
            'right': right_id,
        }

        left_mask = curr_X[:, best_feature] <= best_thresh
        stack.append((curr_X[~left_mask], curr_y[~left_mask], depth + 1, right_id))
        stack.append((curr_X[left_mask], curr_y[left_mask], depth + 1, left_id))
    return tree

### *Predict*

In [27]:
def predict_one(x, tree):
    curr_id = 0
    while 'value' not in tree[curr_id]:
        node = tree[curr_id]
        if x[node['feature']] <= node['threshold']:
            curr_id = node['left']
        else:
            curr_id = node['right']
    return tree[curr_id]['value']

### *Example*

In [33]:
# Data [weight, bark(Yes: 1, No: 0), Ears erect(Yes: 1, No: 0)]
X_data = np.array([
    [3, 0, 1], [4, 0, 1], [2, 0, 1], [5, 0, 0], [3.5, 0, 1], 
    [15, 1, 0], [20, 1, 1], [25, 1, 0], [30, 1, 1], [12, 1, 0], 
    [7, 1, 1], [8, 1, 1], [6, 1, 0], [9, 1, 1], [5.5, 1, 1], 
    [8, 0, 1], [7.5, 0, 0], [9, 0, 1], [10, 0, 0], [6.5, 0, 1]
])

y_data = np.array([
    0, 0, 0, 0, 0, 
    1, 1, 1, 1, 1,
    1, 1, 1, 1, 1, 
    0, 0, 0, 0, 0 
])
 # 0: Cat, 1: Dog

my_tree = train_tree(X_data, y_data)

test_animal = [50, 1, 1]
result = predict_one(test_animal, my_tree)
print(f"Result Predict: {'Dog' if result == 1 else 'Cat'}")

Result Predict: Dog
